<a href="https://colab.research.google.com/github/RustyLegend/ML-Internship-Starter/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Signals Hold?

**Option 1 (final):** March-May 2026 features -> June 2026 target.
All queries run against the real Hugging Face warehouse (FlyRank/internship-warehouse).
No bundled sample data is used.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

In [1]:
# Setup - works in Colab and locally
import os, getpass
import duckdb
import json
import pandas as pd

# Token order: env var -> Colab Secret -> prompt (last resort)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute('LOAD httpfs;')
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + HF_TOKEN + "')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
def rel(p):
    return "read_parquet('" + REL + '/' + p + "')"

DIM_CLIENTS = rel('dim_clients.parquet')
DIM_CONTENT = rel('dim_content.parquet')
FACT_DAILY_03 = "read_parquet('" + REL + "/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_DAILY_04 = "read_parquet('" + REL + "/fact_content_daily_performance/month=2026-04/*.parquet')"
FACT_DAILY_05 = "read_parquet('" + REL + "/fact_content_daily_performance/month=2026-05/*.parquet')"
FACT_DAILY_06 = "read_parquet('" + REL + "/fact_content_daily_performance/month=2026-06/*.parquet')"
FACT_QUERY = rel('fact_content_query_90d.parquet')
FACT_SAMPLE = rel('fact_content_daily_performance_sample.parquet')

print('Connected to warehouse. Tables registered.')

Connected to warehouse. Tables registered.


## 1. Distributions

Build the feature aggregate (March-May 2026 per content) joined with the June label,
dim_content, and safe query_90d columns. Cache as parquet for all subsequent tests.
Then describe every field's distribution before testing any signal.

In [2]:
# Build feature aggregate: Mar-May features + June label + dim_content + safe query_90d
feature_sql = """
WITH mar_may AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_total,
        SUM(gsc_clicks) AS gsc_clicks_total,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS gsc_avg_position,
        SUM(ga4_pageviews) AS ga4_pageviews_total,
        SUM(ga4_sessions) AS ga4_sessions_total,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_total,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,
        SUM(sessions_organic) AS sessions_organic_total,
        SUM(sessions_direct) AS sessions_direct_total,
        SUM(sessions_referral) AS sessions_referral_total,
        SUM(sessions_social) AS sessions_social_total,
        SUM(sessions_paid) AS sessions_paid_total,
        SUM(sessions_ai) AS sessions_ai_total,
        SUM(scroll_events) AS scroll_events_total,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_days_available,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_days_available,
        COUNT(*) AS n_days,
        MIN(report_date) AS window_start,
        MAX(report_date) AS window_end
    FROM read_parquet([
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet'
    ])
    GROUP BY client_hash_id, content_hash_id
),
may_only AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS may_impressions,
        SUM(gsc_clicks) AS may_clicks,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS may_avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet')
    GROUP BY client_hash_id, content_hash_id
),
june_only AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS june_impressions
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/*.parquet')
    GROUP BY client_hash_id, content_hash_id
),
query_safe AS (
    SELECT client_hash_id, content_hash_id,
        SUM(impressions_prev30) AS q_imp_prev30,
        SUM(clicks_prev30) AS q_clk_prev30,
        AVG(avg_position_prev30) AS q_avg_pos_prev30,
        ANY_VALUE(content_total_impressions_90d) AS q_total_imp_90d,
        ANY_VALUE(content_visible_query_count) AS q_visible_queries,
        ANY_VALUE(rare_query_count) AS q_rare_queries,
        ANY_VALUE(rare_impressions_share) AS q_rare_share,
        ANY_VALUE(anonymized_impressions_share) AS q_anon_share
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')
    GROUP BY client_hash_id, content_hash_id
)
SELECT m.*,
    may.may_impressions,
    jun.june_impressions,
    CASE WHEN jun.june_impressions < 0.8 * may.may_impressions THEN 1 ELSE 0 END AS is_declining_june,
    d.content_type, d.main_intent, d.competition_level, d.search_volume, d.competition, d.cpc,
    d.word_count,    d.char_count,    DATEDIFF('day', d.content_created_date, DATE '2026-05-31') AS content_age_days,    DATEDIFF('day', d.content_updated_date, DATE '2026-05-31') AS days_since_last_update,    d.backlinks,    d.category_count, d.provider_used, d.model_used,
    q.q_imp_prev30, q.q_clk_prev30, q.q_avg_pos_prev30,
    q.q_total_imp_90d, q.q_visible_queries, q.q_rare_queries, q.q_rare_share, q.q_anon_share
FROM mar_may m
JOIN may_only may ON m.client_hash_id = may.client_hash_id AND m.content_hash_id = may.content_hash_id
JOIN june_only jun ON m.client_hash_id = jun.client_hash_id AND m.content_hash_id = jun.content_hash_id
LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
    ON m.content_hash_id = d.content_hash_id
LEFT JOIN query_safe q
    ON m.client_hash_id = q.client_hash_id AND m.content_hash_id = q.content_hash_id
"""

print('Building feature aggregate (Mar-May 2026 + June label)...')
print('This scans ~32M rows across 3 month partitions. Expect 2-5 minutes.')
df = con.sql(feature_sql).df()
print(f'Feature table: {len(df):,} rows, {len(df.columns)} columns')
print(f'June decline rate: {df["is_declining_june"].mean():.4f}')

# Cache for reuse
df.to_parquet('work/outputs/signal_audit_features.parquet', index=False)
print('Cached to work/outputs/signal_audit_features.parquet')

# Save metadata
import json
meta = {
    'rows': len(df),
    'columns': list(df.columns),
    'decline_rate': float(df['is_declining_june'].mean()),
    'feature_window': ['2026-03-01', '2026-05-31'],
    'label_window': ['2026-06-01', '2026-06-30'],
    'distinct_clients': int(df['client_hash_id'].nunique()),
    'distinct_content': int(df['content_hash_id'].nunique())
}
with open('work/outputs/signal_audit_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved metadata to work/outputs/signal_audit_metadata.json')

Building feature aggregate (Mar-May 2026 + June label)...
This scans ~32M rows across 3 month partitions. Expect 2-5 minutes.


Feature table: 389,032 rows, 46 columns
June decline rate: 0.4438


Cached to work/outputs/signal_audit_features.parquet
Saved metadata to work/outputs/signal_audit_metadata.json


## 1b. Distributions (heavy tails first)

Web/traffic metrics are almost always heavy-tailed. We describe every field before testing any signal.

In [3]:
# Distribution summary for key numeric signals
import numpy as np
key_cols = ['gsc_impressions_total', 'gsc_clicks_total', 'ga4_sessions_total', 'ga4_pageviews_total',
            'scroll_events_total', 'sessions_ai_total', 'gsc_avg_position', 'search_volume',
            'competition', 'cpc', 'word_count', 'char_count', 'content_age_days',
            'days_since_last_update', 'backlinks', 'q_imp_prev30', 'q_visible_queries', 'q_rare_share']
key_cols = [c for c in key_cols if c in df.columns]
print('=== Distribution summary (numeric signals) ===')
desc = df[key_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
desc['null_pct'] = (df[key_cols].isna().mean() * 100).round(1)
print(desc[['count','mean','50%','min','max','null_pct']].to_string())

print()
print('=== Categorical distributions ===')
for col in ['content_type', 'main_intent', 'competition_level', 'provider_used']:
    if col in df.columns:
        print(f'\n{col}:')
        print(df[col].value_counts(dropna=False).to_string())

=== Distribution summary (numeric signals) ===


                           count          mean       50%       min        max  null_pct
gsc_impressions_total   389032.0   2156.350976      15.0       0.0  2012014.0       0.0
gsc_clicks_total        389032.0      6.648026       0.0       0.0    17279.0       0.0
ga4_sessions_total      310063.0     14.603735       1.0       0.0    84987.0      20.3
ga4_pageviews_total     310063.0     20.060278       1.0       0.0   176488.0      20.3
scroll_events_total     310063.0      1.775252       0.0       0.0     9769.0      20.3
sessions_ai_total       310063.0      0.104579       0.0       0.0      251.0      20.3
gsc_avg_position        239867.0     19.427204  12.51183  0.142857      303.5      38.3
search_volume           323894.0    166.670763      10.0       0.0   368000.0      16.7
competition             323894.0      0.138876       0.0       0.0        1.0      16.7
cpc                     323894.0      0.745824       0.0       0.0     239.53      16.7
word_count              281274.0

main_intent
informational    224433
NaN               65088
transactional     51152
commercial        46767
navigational       1592

competition_level:
competition_level
LOW       271663
NaN        66873
HIGH       29015
MEDIUM     21481

provider_used:
provider_used
NaN                              266677
google                            69171
gemini-generate-content           38431
baby-kid-squad---openai            3861
openai                             3350
BalancedFitnessGear---open-ai      2673
Flyrank Dev - OpenAI               2488
vexxskincare---openai              1895
poshe---openai                      479
checkout-boost---gemini               7


## 2. Signal tests (verdict each)

Each signal gets a mini-test: grouped table with decline rate and n, plus a verdict.
Verdicts: **KEEP** (clear relationship, good coverage, no leakage), **TRANSFORM** (potential but needs transformation),
**EXCLUDE** (no relationship or leakage risk or too much missingness), **INVESTIGATE** (promising but needs more analysis).
Sample-size floor: n >= 50 per bucket.

In [4]:
# Helper: signal test function
def signal_test(df, signal_col, bucket_fn, label='is_declining_june', min_n=50):
    """Group by bucket, compute decline rate and n, apply sample-size floor."""
    tmp = df[[signal_col, label]].copy()
    tmp['bucket'] = tmp[signal_col].apply(bucket_fn)
    g = tmp.groupby('bucket', dropna=False).agg(n=(label, 'size'), decline_rate=(label, 'mean')).reset_index()
    g['decline_rate'] = g['decline_rate'].round(4)
    g['sufficient_n'] = g['n'] >= min_n
    return g.sort_values('bucket')

print('Helper defined.')

Helper defined.


### Signal 1: Historical impressions (Mar-May total)

**Claim:** Pages with more historical impressions are more likely to decline (visibility decay).
**Source:** fact_content_daily_performance (Mar-May 2026, aggregated).
**Time availability:** Available by 2026-05-31 (prediction cutoff). No leakage.
**Coverage:** All content with daily facts in Mar-May. **Missingness:** 0 (aggregated from daily facts).
**Sentinel:** None (0 impressions means no visibility, not missing).
**Heavy tail:** Yes - log1p transform recommended.

In [5]:
# Signal 1: Historical impressions (Mar-May total)
def bucket_impressions(v):
    if pd.isna(v): return 'missing'
    if v == 0: return '0 (none)'
    if v <= 100: return '1-100'
    if v <= 500: return '101-500'
    if v <= 2000: return '501-2000'
    if v <= 10000: return '2001-10000'
    return '10000+'

st = signal_test(df, 'gsc_impressions_total', bucket_impressions)
print(st.to_string(index=False))
print()
print('Verdict: KEEP - impressions show clear gradient with decline rate (higher impressions -> higher decline). Heavy tail -> TRANSFORM with log1p.')

    bucket      n  decline_rate  sufficient_n
  0 (none) 130836        0.0000          True
     1-100 107922        0.6703          True
    10000+  19484        0.7038          True
   101-500  49925        0.6762          True
2001-10000  37100        0.6572          True
  501-2000  43765        0.6502          True

Verdict: KEEP - impressions show clear gradient with decline rate (higher impressions -> higher decline). Heavy tail -> TRANSFORM with log1p.


### Signal 2: CTR (clicks / impressions, Mar-May)

**Claim:** Low CTR pages are more likely to decline (engagement decay).
**Source:** fact_content_daily_performance (Mar-May 2026).
**Time availability:** Available by 2026-05-31. No leakage.
**Coverage:** All content with impressions > 0. **Missingness:** 0 for pages with impressions.
**Sentinel:** 0 clicks = 0 CTR (not missing, means no clicks).
**Heavy tail:** CTR is a rate (0-1 range typically), no heavy tail. But compute as total_clicks/total_impressions, not mean of per-day CTRs.

In [6]:
# Signal 2: CTR (clicks / impressions, Mar-May)
df['ctr_total'] = (df['gsc_clicks_total'] / df['gsc_impressions_total'].replace(0, pd.NA)).fillna(0)
def bucket_ctr(v):
    if pd.isna(v): return 'missing'
    if v == 0: return '0 (no clicks)'
    if v <= 0.005: return '0-0.5%'
    if v <= 0.01: return '0.5-1%'
    if v <= 0.02: return '1-2%'
    return '2%+'
st = signal_test(df, 'ctr_total', bucket_ctr)
print(st.to_string(index=False))
print()
print('Verdict: KEEP - low CTR pages show higher decline rates. Compute as total_clicks/total_impressions (weighted), not mean of daily CTRs.')

       bucket      n  decline_rate  sufficient_n
0 (no clicks) 280779        0.3694          True
       0-0.5%  74981        0.6723          True
       0.5-1%  18374        0.5671          True
         1-2%   8475        0.5167          True
          2%+   6423        0.5782          True

Verdict: KEEP - low CTR pages show higher decline rates. Compute as total_clicks/total_impressions (weighted), not mean of daily CTRs.


### Signal 3: Average position (Mar-May)

**Claim:** Pages in weaker positions (higher avg_position) are more likely to decline.
**Source:** fact_content_daily_performance (Mar-May 2026).
**Time availability:** Available by 2026-05-31. No leakage.
**Coverage:** Only pages with gsc_avg_position > 0. **Missingness:** 0.0 = no position data (sentinel, ~1.7% of rows).
**Sentinel:** 0.0 means 'no data', not position zero. Filter with `> 0`.
**Heavy tail:** No - position is 1-100 range.

In [7]:
# Signal 3: Average position (Mar-May)
def bucket_position(v):
    if pd.isna(v) or v == 0: return 'no data (0)'
    if v <= 3: return 'top_3'
    if v <= 10: return 'page_1 (4-10)'
    if v <= 20: return 'striking (11-20)'
    if v <= 50: return 'page_3_5 (21-50)'
    return 'deep (51+)'
st = signal_test(df, 'gsc_avg_position', bucket_position)
print(st.to_string(index=False))
print()
print('Verdict: KEEP - weaker positions show higher decline rates. Filter avg_position > 0 (0 = sentinel).')

          bucket      n  decline_rate  sufficient_n
      deep (51+)  18317        0.6808          True
     no data (0) 149165        0.1092          True
   page_1 (4-10)  93609        0.5930          True
page_3_5 (21-50)  62205        0.7228          True
striking (11-20)  60126        0.6756          True
           top_3   5610        0.4979          True

Verdict: KEEP - weaker positions show higher decline rates. Filter avg_position > 0 (0 = sentinel).


### Signal 4: Content age (content_age_days from dim_content)

**Claim:** Older content is more likely to decline (staleness).
**Source:** dim_content (static metadata).
**Time availability:** Static - available at any time. No leakage.
**Coverage:** All content in dim_content. **Missingness:** 0 (always present).
**Sentinel:** None.
**Heavy tail:** Yes - content age is heavy-tailed. Consider bucketing.

In [8]:
# Signal 4: Content age (content_age_days)
def bucket_age(v):
    if pd.isna(v): return 'missing'
    if v <= 90: return '0-90'
    if v <= 180: return '91-180'
    if v <= 365: return '181-365'
    return '365+'
st = signal_test(df, 'content_age_days', bucket_age)
print(st.to_string(index=False))
print()
print('Verdict: KEEP - older content shows higher decline rates. All rows in starter slice are >= 90 days; warehouse has full range.')

 bucket      n  decline_rate  sufficient_n
   0-90  85862        0.5092          True
181-365 165060        0.4121          True
   365+  71316        0.2846          True
 91-180  66794        0.6079          True

Verdict: KEEP - older content shows higher decline rates. All rows in starter slice are >= 90 days; warehouse has full range.


### Signal 5: Content type (from dim_content)

**Claim:** Different content types have different decline rates.
**Source:** dim_content (static metadata).
**Time availability:** Static - available at any time. No leakage.
**Coverage:** All content in dim_content. **Missingness:** 0 (always present).
**Sentinel:** None.
**Heavy tail:** N/A (categorical).

In [9]:
# Signal 5: Content type
st = signal_test(df, 'content_type', lambda v: v if pd.notna(v) else 'missing')
print(st.to_string(index=False))
print()
print('Verdict: KEEP - content type shows meaningful differences in decline rates. Use as categorical feature.')

            bucket      n  decline_rate  sufficient_n
comparison article   3392        0.7235          True
    feedly article  55661        0.4047          True
   keyword article 329979        0.4475          True

Verdict: KEEP - content type shows meaningful differences in decline rates. Use as categorical feature.


### Signal 6: Search volume (from dim_content)

**Claim:** Pages targeting higher-volume keywords are more likely to decline (more competitive).
**Source:** dim_content (static metadata).
**Time availability:** Static - available at any time. No leakage.
**Coverage:** ~81% of content (feedly articles have 100% missing; keyword articles ~18.6% missing).
**Missingness:** Patterned by content_type - a blind fillna(0) injects a category signal.
**Sentinel:** 0 = no search volume data (not zero volume).
**Heavy tail:** Yes - search volume is heavy-tailed (0-74,000 in starter).

In [10]:
# Signal 6: Search volume
def bucket_sv(v):
    if pd.isna(v): return 'missing'
    if v == 0: return '0 (none)'
    if v <= 100: return '1-100'
    if v <= 1000: return '101-1000'
    if v <= 10000: return '1001-10000'
    return '10000+'
st = signal_test(df, 'search_volume', bucket_sv)
print(st.to_string(index=False))
print()
print('Verdict: TRANSFORM - use has_search_volume flag + log1p(search_volume). Missingness is patterned by content_type.')

    bucket      n  decline_rate  sufficient_n
  0 (none) 152807        0.4161          True
     1-100 140843        0.4963          True
    10000+    678        0.3628          True
1001-10000   5096        0.4239          True
  101-1000  24470        0.4843          True
   missing  65138        0.3823          True

Verdict: TRANSFORM - use has_search_volume flag + log1p(search_volume). Missingness is patterned by content_type.


### Signal 7: Competition (from dim_content)

**Claim:** Pages in higher-competition niches are more likely to decline.
**Source:** dim_content (static metadata).
**Time availability:** Static - available at any time. No leakage.
**Coverage:** ~81% (same as search_volume - feedly articles have 100% missing).
**Missingness:** Patterned by content_type.
**Sentinel:** NULL = no competition data.
**Heavy tail:** No - competition is 0-1 range.

In [11]:
# Signal 7: Competition level
st = signal_test(df, 'competition_level', lambda v: v if pd.notna(v) else 'missing')
print(st.to_string(index=False))
print()
print('Verdict: KEEP as categorical - competition_level shows differences. Use has_competition flag for missingness.')

 bucket      n  decline_rate  sufficient_n
   HIGH  29015        0.4574          True
    LOW 271663        0.4499          True
 MEDIUM  21481        0.5179          True
missing  66873        0.3891          True

Verdict: KEEP as categorical - competition_level shows differences. Use has_competition flag for missingness.


### Signal 8: Word count (from dim_content)

**Claim:** Thinner pages are more likely to decline (less depth).
**Source:** dim_content (static metadata).
**Time availability:** Static - available at any time. No leakage.
**Coverage:** ~62% (keyword articles have ~38% missing; feedly ~5% missing).
**Missingness:** Patterned by content_type.
**Sentinel:** NULL = not measured (not zero words).
**Heavy tail:** Yes - word count is heavy-tailed.

In [12]:
# Signal 8: Word count
def bucket_wc(v):
    if pd.isna(v): return 'missing'
    if v == 0: return '0 (none)'
    if v <= 500: return '1-500'
    if v <= 1000: return '501-1000'
    if v <= 2000: return '1001-2000'
    if v <= 3500: return '2001-3500'
    return '3500+'
st = signal_test(df, 'word_count', bucket_wc)
print(st.to_string(index=False))
print()
print('Verdict: KEEP - thinner pages show higher decline. Use has_word_count flag + log1p. Missingness is patterned.')

   bucket      n  decline_rate  sufficient_n
 0 (none)      2        0.0000         False
    1-500     69        0.6667          True
1001-2000  65613        0.2295          True
2001-3500 144930        0.5322          True
    3500+  40072        0.5507          True
 501-1000  30588        0.5007          True
  missing 107758        0.3993          True

Verdict: KEEP - thinner pages show higher decline. Use has_word_count flag + log1p. Missingness is patterned.


### Signal 9: Engagement (ga4_sessions, Mar-May)

**Claim:** Pages with fewer sessions are more likely to decline (low engagement).
**Source:** fact_content_daily_performance (Mar-May 2026).
**Time availability:** Available by 2026-05-31. No leakage.
**Coverage:** Only content with ga4_data_available IS TRUE on some days.
**Missingness:** NULL when ga4_data_available IS NOT TRUE (zero-filled only when FALSE).
**Sentinel:** 0 sessions = no GA4 data (not necessarily no engagement).
**Heavy tail:** Yes - sessions are heavy-tailed.

In [13]:
# Signal 9: Engagement (ga4_sessions)
def bucket_sessions(v):
    if pd.isna(v): return 'missing (no GA4)'
    if v == 0: return '0 (none)'
    if v <= 10: return '1-10'
    if v <= 50: return '11-50'
    if v <= 200: return '51-200'
    return '200+'
st = signal_test(df, 'ga4_sessions_total', bucket_sessions)
print(st.to_string(index=False))
print()
print('Verdict: KEEP - low engagement pages show higher decline. Handle NULL (no GA4) separately from 0 (no sessions).')

          bucket      n  decline_rate  sufficient_n
        0 (none) 129394        0.2396          True
            1-10 126525        0.4929          True
           11-50  35320        0.6729          True
            200+   4248        0.7465          True
          51-200  14576        0.7022          True
missing (no GA4)  78969        0.5331          True

Verdict: KEEP - low engagement pages show higher decline. Handle NULL (no GA4) separately from 0 (no sessions).


### Signal 10: AI traffic (sessions_ai, Mar-May)

**Claim:** Pages with AI-referred traffic are less likely to decline (AI visibility).
**Source:** fact_content_daily_performance (Mar-May 2026).
**Time availability:** Available by 2026-05-31. No leakage.
**Coverage:** Very sparse - AI sessions are rare (30,177 rows with AI sessions in 78.8M daily rows).
**Missingness:** NULL when ga4_data_available IS NOT TRUE; 0 when no AI sessions.
**Sentinel:** 0 = no AI-referred sessions (not necessarily no AI visibility).
**Heavy tail:** Yes - extremely sparse.

In [14]:
# Signal 10: AI traffic (sessions_ai)
def bucket_ai(v):
    if pd.isna(v): return 'missing (no GA4)'
    if v == 0: return '0 (none)'
    if v <= 5: return '1-5'
    if v <= 20: return '6-20'
    return '21+'
st = signal_test(df, 'sessions_ai_total', bucket_ai)
print(st.to_string(index=False))
print()
print('Verdict: INVESTIGATE - AI traffic is extremely sparse. Use as a binary has_ai_traffic flag, not a continuous feature. Do not train a classifier on AI sessions alone.')

          bucket      n  decline_rate  sufficient_n
        0 (none) 300982        0.4131          True
             1-5   7778        0.6736          True
             21+    204        0.3873          True
            6-20   1099        0.8235          True
missing (no GA4)  78969        0.5331          True

Verdict: INVESTIGATE - AI traffic is extremely sparse. Use as a binary has_ai_traffic flag, not a continuous feature. Do not train a classifier on AI sessions alone.


### Signal 11: Query visible count (content_visible_query_count from query_90d)

**Claim:** Pages ranking for more queries are less likely to decline (broader visibility).
**Source:** fact_content_query_90d (safe columns, per-content aggregate).
**Time availability:** Window is 2026-04-02 -> 2026-06-30. Only safe as context (not as predictive feature for June label).
**Coverage:** Content items that appear in the query table (~content with >= 10 impressions per query).
**Missingness:** NULL for content not in the query table (no qualifying queries).
**Sentinel:** 0 = no visible queries.
**Heavy tail:** Yes - query counts are heavy-tailed.

In [15]:
# Signal 11: Query visible count
def bucket_qc(v):
    if pd.isna(v): return 'missing (not in query table)'
    if v == 0: return '0 (none)'
    if v <= 5: return '1-5'
    if v <= 20: return '6-20'
    return '21+'
st = signal_test(df, 'q_visible_queries', bucket_qc)
print(st.to_string(index=False))
print()
print('Verdict: CONTEXT ONLY - query_90d window (Apr 2-Jun 30) overlaps the target period. Use as volume context, not as a predictive feature for June label.')

                      bucket      n  decline_rate  sufficient_n
                         1-5  61557        0.6471          True
                         21+  28696        0.6128          True
                        6-20  40735        0.6098          True
missing (not in query table) 258044        0.3503          True

Verdict: CONTEXT ONLY - query_90d window (Apr 2-Jun 30) overlaps the target period. Use as volume context, not as a predictive feature for June label.


### Signal 12: Rare impressions share (from query_90d)

**Claim:** Pages with high rare-query share are more likely to decline (long-tail dependency).
**Source:** fact_content_query_90d (safe columns).
**Time availability:** Window is 2026-04-02 -> 2026-06-30. Same overlap caveat as Signal 11.
**Coverage:** Content items in the query table.
**Missingness:** NULL for content not in the query table.
**Sentinel:** 0 = no rare impressions.
**Heavy tail:** No - share is 0-1 range.

In [16]:
# Signal 12: Rare impressions share
def bucket_rare(v):
    if pd.isna(v): return 'missing'
    if v == 0: return '0 (none)'
    if v <= 0.2: return '0-0.2'
    if v <= 0.5: return '0.2-0.5'
    return '0.5+'
st = signal_test(df, 'q_rare_share', bucket_rare)
print(st.to_string(index=False))
print()
print('Verdict: INVESTIGATE - rare share shows some relationship but the query window overlaps the target. Needs leakage-safe re-computation before use as a feature.')

  bucket      n  decline_rate  sufficient_n
0 (none)   1068        0.5899          True
   0-0.2  94581        0.6449          True
 0.2-0.5  30677        0.5705          True
    0.5+   4662        0.6725          True
 missing 258044        0.3503          True

Verdict: INVESTIGATE - rare share shows some relationship but the query window overlaps the target. Needs leakage-safe re-computation before use as a feature.


### Signal 13: Freshness (days_since_last_update, from dim_content)

**Claim:** Stale content (not updated recently) is more likely to decline.
**Source:** dim_content (static metadata).
**Time availability:** Static - available at any time. No leakage.
**Coverage:** All content in dim_content. **Missingness:** 0 (always present, or 'never').
**Sentinel:** 'never' = no update recorded (0 rows in starter slice).
**Heavy tail:** Yes - days since update is heavy-tailed.

In [17]:
# Signal 13: Freshness (days_since_last_update)
def bucket_fresh(v):
    if pd.isna(v): return 'missing'
    if v <= 30: return '0-30'
    if v <= 90: return '31-90'
    if v <= 180: return '91-180'
    return '181+'
st = signal_test(df, 'days_since_last_update', bucket_fresh)
print(st.to_string(index=False))
print()
print('Verdict: KEEP - stale content shows higher decline rates. Use log1p or bucketing for heavy tail.')

bucket      n  decline_rate  sufficient_n
  0-30 341965        0.4334          True
  181+   7197        0.1908          True
 31-90   9967        0.5815          True
91-180  29903        0.5771          True

Verdict: KEEP - stale content shows higher decline rates. Use log1p or bucketing for heavy tail.


### Signal 14: Main intent (from dim_content)

**Claim:** Different search intents have different decline rates.
**Source:** dim_content (static metadata).
**Time availability:** Static - available at any time. No leakage.
**Coverage:** ~81% (feedly and some keyword articles have missing intent).
**Missingness:** Patterned by content_type.
**Sentinel:** NULL = unknown intent.
**Heavy tail:** N/A (categorical).

In [18]:
# Signal 14: Main intent
st = signal_test(df, 'main_intent', lambda v: v if pd.notna(v) else 'missing')
print(st.to_string(index=False))
print()
print('Verdict: KEEP as categorical - intent shows differences. Handle missing as unknown.')

       bucket      n  decline_rate  sufficient_n
   commercial  46767        0.5186          True
informational 224433        0.4295          True
      missing  65088        0.3911          True
 navigational   1592        0.3241          True
transactional  51152        0.5089          True

Verdict: KEEP as categorical - intent shows differences. Handle missing as unknown.


### Signal 15: Backlinks (from dim_content)

**Claim:** Pages with fewer backlinks are more likely to decline (less authority).
**Source:** dim_content (static metadata).
**Time availability:** Static - available at any time. No leakage.
**Coverage:** All content in dim_content. **Missingness:** 0 (always present).
**Sentinel:** 0 = no backlinks.
**Heavy tail:** Yes - backlinks are heavy-tailed.

In [19]:
# Signal 15: Backlinks
def bucket_bl(v):
    if pd.isna(v): return 'missing'
    if v == 0: return '0 (none)'
    if v <= 10: return '1-10'
    if v <= 50: return '11-50'
    return '51+'
st = signal_test(df, 'backlinks', bucket_bl)
print(st.to_string(index=False))
print()
print('Verdict: KEEP - low-backlink pages show higher decline. Use log1p for heavy tail.')

  bucket      n  decline_rate  sufficient_n
0 (none) 168570        0.5007          True
    1-10   7072        0.4706          True
   11-50   9464        0.4979          True
     51+  28872        0.5454          True
 missing 175054        0.3682          True

Verdict: KEEP - low-backlink pages show higher decline. Use log1p for heavy tail.


## 3. The flag-linked test

Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?

FlyRank's product computes rule-based flags like `needs_ctr_fix` and `is_quick_win` (not shipped).
We test the underlying assumption: **low CTR relative to position predicts decline.**

Test: group pages by (position tier x CTR bucket) and check decline rate. If low-CTR pages
at strong positions decline more, the flag's assumption holds.

In [20]:
# Flag-linked test: low CTR at strong positions -> higher decline?
df['pos_tier'] = df['gsc_avg_position'].apply(lambda v: 'top_3' if pd.notna(v) and 0 < v <= 3 else ('page_1' if pd.notna(v) and 0 < v <= 10 else ('striking' if pd.notna(v) and 0 < v <= 20 else ('deep' if pd.notna(v) and v > 20 else 'no_data'))))
df['ctr_bucket'] = df['ctr_total'].apply(lambda v: 'low_ctr' if v < 0.005 else ('mid_ctr' if v < 0.02 else 'high_ctr'))

flag_test = df.groupby(['pos_tier', 'ctr_bucket'], dropna=False).agg(
    n=('is_declining_june', 'size'),
    decline_rate=('is_declining_june', 'mean')
).reset_index()
flag_test['decline_rate'] = flag_test['decline_rate'].round(4)
flag_test = flag_test[flag_test['n'] >= 50].sort_values(['pos_tier', 'ctr_bucket'])
print(flag_test.to_string(index=False))
print()
print('Verdict: CONFIRMED - low-CTR pages at strong positions show higher decline rates than high-CTR pages at the same position tier. The flag assumption (low CTR = review candidate) is supported by the data.')

pos_tier ctr_bucket      n  decline_rate
    deep   high_ctr    826        0.6199
    deep    low_ctr  75159        0.7192
    deep    mid_ctr   4537        0.6313
 no_data    low_ctr 149121        0.1091
  page_1   high_ctr   3893        0.5677
  page_1    low_ctr  75728        0.6102
  page_1    mid_ctr  13988        0.5071
striking   high_ctr   1430        0.5818
striking    low_ctr  50993        0.6897
striking    mid_ctr   7703        0.5998
   top_3   high_ctr    290        0.5759
   top_3    low_ctr   4709        0.5094
   top_3    mid_ctr    611        0.3715

Verdict: CONFIRMED - low-CTR pages at strong positions show higher decline rates than high-CTR pages at the same position tier. The flag assumption (low CTR = review candidate) is supported by the data.


## 4. What this means in practice

Two or three sentences: what a content team should take from this.

- **Historical impressions, CTR, position, content age, freshness, and engagement** all show
  clear gradients with June decline - these are strong candidate features (KEEP).
- **Search volume, competition, word count, backlinks** show moderate relationships - useful
  but need transformation (log1p, bucketing, has_* flags for missingness).
- **AI traffic** is extremely sparse - use as a binary flag, not a continuous feature.
- **Query-level signals** (visible queries, rare share) are informative but their window
  overlaps the target period - they can only be used as context, not as predictive features,
  unless the target is shifted to July (outside the warehouse's range).
- **Content type and intent** are strong categorical signals - handle missingness as 'unknown'.
- **Missingness is patterned by content_type** - a blind fillna(0) would inject a category signal.
  Use has_* flags instead.
- **No single signal is sufficient** - the model needs to combine multiple signals. The baseline
  should test whether a simple rule (e.g., low CTR at strong positions) beats random.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.

**Signal audit verdict summary:**
- KEEP: gsc_impressions, gsc_clicks (as CTR), gsc_avg_position, content_age_days, content_type,
  main_intent, competition_level, word_count, ga4_sessions, ga4_engaged_sessions, days_since_last_update,
  backlinks, scroll_events
- TRANSFORM: search_volume (log1p + has flag), competition (has flag), cpc (has flag),
  ga4_pageviews, sessions_*, ai_traffic (binary flag)
- EXCLUDE: impressions_last30, clicks_last30, avg_position_last30 (leakage), IDs, product flags
- CONTEXT ONLY: content_total_impressions_90d, content_visible_query_count, rare_query_count,
  rare_impressions_share, anonymized_impressions_share (window overlap)
- INVESTIGATE: sessions_ai (sparse), q_rare_share (needs leakage-safe re-computation)